# Primeira rede neural com Keras

Do perceptron isolado até um modelo compilado e pronto para treinar,
passando pelas duas APIs de construção do Keras: **Sequential** e **Funcional**.

## [1] Importação de bibliotecas

In [ ]:
"""
Define uma seed global para garantir reprodutibilidade máxima
no treinamento com TensorFlow/Keras.

NOTE 1: Para aulas, deixo essa parte comentada para demonstrar o
        comportamento estocástico das práticas com DL.
NOTE 2: Na prática diária com DL, determinamos as SEEDs somente no 
        split dos dados, pois operações em GPU sofrem perda de 
        performance quando forçadas ao determinismo.
NOTE 3: Em alguns casos de treinamento avançado com DL, não é 
        possível estabelecer 100% de determinismo devido à alta
        dependência de operações paralelizadas.
"""
# 1. Variáveis de ambiente (precisam ser definidas ANTES de importar TF!
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      # força kernels determinísticos
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'    # determinismo em GPU (cuDNN)


import tensorflow as tf
import keras
import numpy as np
from absl import logging

# seeds determinadas após importação dos módulos
keras.utils.set_random_seed(SEED)
# 3. Habilita operações determinísticas no TF (TF >= 2.8)
# NOTE: Haverá perda de performance
# NOTE: Em alguns casos, ainda não será 100% determinístico (paralelismo!)
tf.config.experimental.enable_op_determinism()

# silenciar alguns avisos do tf que não serão relevantes
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

## [2] Função utilitária de visualização

Sempre que criamos um modelo, queremos ver seu resumo (camadas, formatos
e número de parâmetros) e seu diagrama. Uma função evita repetir as
mesmas duas linhas para cada modelo construído neste notebook.

In [ ]:
def describe_model(model):
    model.summary()
    return keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

## [3] Uma camada isolada (perceptrons)

Uma camada `Dense` com N unidades representa N perceptrons em paralelo,
cada um recebendo as mesmas entradas (features).

In [ ]:
FEATURES = 2  # quantidade de entradas (x1, x2)
UNITS = 4     # quantidade de perceptrons na camada

layer = keras.layers.Dense(UNITS, activation='relu', name='layer1')

# a camada só é inicializada quando conhece o formato da entrada
# ("declaração preguiçosa"). Aqui forçamos essa inicialização manualmente.
# None representa o eixo de batch (explicado nas próximas aulas!).
layer.build((None, FEATURES))

print("\n ================ \n")
print('pesos das entradas (X):')
print(layer.weights[0].numpy())  # índice 0 = pesos das entradas (Kernel)
print("\n ================ \n")
print('pesos dos vieses (bias):')
print(layer.weights[1].numpy())  # índice 1 = pesos dos vieses

## [4] Testando a camada com uma entrada

In [ ]:
inputs = np.random.random((1, FEATURES))
print("INPUTS:", inputs)

print("OUTPUTS:", layer(inputs))

## [5] Modelo sequencial (API Sequential)

Duas formas equivalentes de empilhar camadas: adicionando uma a uma com
`.add()`, ou passando uma lista de camadas diretamente ao `Sequential`.

In [ ]:
model = keras.Sequential()
model.add(keras.layers.Input(shape=(2,), name='inputs'))
model.add(keras.layers.Dense(4, activation='relu', name='hidden1'))
model.add(keras.layers.Dense(4, activation='relu', name='hidden2'))
model.add(keras.layers.Dense(4, activation='relu', name='hidden3'))
model.add(keras.layers.Dense(1, activation='sigmoid', name='output'))

In [ ]:
# forma equivalente (mesmo resultado, sintaxe mais compacta)
model = keras.Sequential([
    keras.layers.Input(shape=(2,), name='inputs'),
    keras.layers.Dense(4, activation='relu', name='hidden1'),
    keras.layers.Dense(4, activation='relu', name='hidden2'),
    keras.layers.Dense(4, activation='relu', name='hidden3'),
    keras.layers.Dense(1, activation='sigmoid', name='output')
])

describe_model(model)

In [ ]:
model.weights

## [6] Modelo com a API Funcional (Functional API)

A API Funcional trata cada camada como uma função aplicada sobre a
camada anterior, permitindo topologias mais flexíveis que o Sequential.

In [ ]:
inputs = keras.layers.Input(shape=(3,), name='inputs')
x = keras.layers.Dense(4, activation='relu', name='hidden1')(inputs)
x = keras.layers.Dense(4, activation='relu', name='hidden2')(x)
x = keras.layers.Dense(4, activation='relu', name='hidden3')(x)
x = keras.layers.Dense(4, activation='relu', name='hidden4')(x)
outputs = keras.layers.Dense(1, activation='sigmoid', name='output')(x)

model = keras.Model(inputs, outputs)

describe_model(model)

## [7] Predição com pesos aleatórios (sem treino)

Como o modelo ainda não foi treinado, a predição reflete apenas os
pesos aleatórios inicializados — ainda não aprendeu nada.

In [ ]:
inputs = np.random.random((1, 3)).astype('float32')
y_pred = model.predict(inputs)

print("INPUTS:", inputs)
print("OUTPUTS: ", y_pred)

## [8] Compilando o modelo para treino

A compilação define o otimizador, a função de perda e as métricas
usadas durante o treinamento.

In [ ]:
model.compile(
    optimizer='SGD',
    loss='binary_crossentropy',
    metrics=['accuracy']
)